# Makeup and skincare dupe recommendations: ingredients, texture and price

Run all cells from the project directory. Recommendations are candidates, not proof of equivalent performance. Parser joins and texture extraction are heuristics; inspect the exported review files. The source includes makeup and skincare products. Check review availability separately for each category; prices and stock are historical. Ingredient order is a concentration proxy, not a measured concentration.

In [1]:
import ast, html, re, unicodedata
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer, TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
from IPython.display import display

DATA_DIR = Path('data')
df = pd.read_csv(DATA_DIR / 'product_info.csv')
CATEGORIES = {'Cleansers', 'Eye Care', 'Lip Balms & Treatments', 'Masks', 'Moisturizers', 'Self Tanners', 'Sunscreen', 'Treatments'}
MAKEUP_TYPES = {'Foundation', 'Setting Spray & Powder', 'Face Primer', 'Highlighter', 'Concealer', 'Contour', 'Tinted Moisturizer', 'BB & CC Cream', 'Color Correct', 'Mascara', 'Eyebrow', 'Eyeliner', 'Eyeshadow', 'Eye Primer', 'Under-Eye Concealer', 'Blush', 'Bronzer', 'Lipstick', 'Lip Balm & Treatment', 'Lip Gloss', 'Lip Liner', 'Liquid Lipstick', 'Lip Plumper', 'Lip Stain'}
products = df[df.ingredients.notna()].copy().reset_index(drop=True)
print('Products with ingredient fields:', len(products))
print('Missing highlights among them:', products.highlights.isna().sum())

Products with ingredient fields: 7549
Missing highlights among them: 1729


## Highlights and ingredients
Missing highlights become an empty list. Multi-section lists with labels or possible separate formulas are held for review. Joined fragments are also exported for checking.

In [2]:
def highlights(value):
    if not isinstance(value, (str, list)): return []
    if isinstance(value, str):
        try: value = ast.literal_eval(value)
        except (SyntaxError, ValueError): value = [value]
    if not isinstance(value, list): return []
    marketing = {'community favorite', 'clean at sephora', 'clean + planet positive', 'black owned at sephora'}
    mapping = {'best for dry skin':['skin_dry'], 'best for oily skin':['skin_oily'], 'best for normal skin':['skin_normal'], 'best for combination skin':['skin_combination'], 'best for dry, combo, normal skin':['skin_dry','skin_combination','skin_normal'], 'best for oily, combo, normal skin':['skin_oily','skin_combination','skin_normal']}
    out=[]
    for item in value:
        if not isinstance(item, str): continue
        s = re.sub(r'\s+', ' ', re.sub(r'[-‐‑–—]', ' ', unicodedata.normalize('NFKC', item).casefold())).strip()
        if not s or s in marketing or s.startswith('allure '): continue
        out.extend(mapping.get(s, [s.replace(' ', '_')]))
    return list(dict.fromkeys(out))

def base_normalize(s):
    s = unicodedata.normalize('NFKC', html.unescape(s)).casefold()
    s = re.sub(r'[‐‑‒–—−]', '-', s)
    s = re.sub(r'\s*/\s*', '/', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return re.sub(r'[\s*†‡]+$', '', s).strip(' .;')

ALIASES = {'aqua/water/eau':'water', 'water/aqua/eau':'water', 'aqua (water)':'water', 'water (aqua)':'water', 'hydroxyacetophe-none':'hydroxyacetophenone', 'pan-thenol':'panthenol'}
def normalize(s):
    s = base_normalize(s)
    s = re.sub(r'\s*\(\s*\d+(?:\.\d+)?\s*%\s*\)', '', s)
    s = re.sub(r'\s+\d+(?:\.\d+)?\s*%', '', s).strip()
    return ALIASES.get(s,s)

def split_formula(text):
    out, part, stack = [], [], []
    for i, ch in enumerate(text):
        if ch in '([': stack.append(ch)
        elif ch in ')]':
            expected = '(' if ch == ')' else '['
            if not stack or stack.pop() != expected: raise ValueError('Unbalanced delimiters')
        # Protect numeric chemical prefixes, e.g. 1,2-hexanediol.
        numeric = ch == ',' and bool(re.search(r'(?:^|[\s,(])\d+$', text[:i])) and bool(re.match(r'\d+(?:,\d+)*-', text[i+1:]))
        if ch == ',' and not stack and not numeric:
            item = base_normalize(''.join(part))
            if item: out.append(item)
            part=[]
        else: part.append(ch)
    if stack: raise ValueError('Unbalanced delimiters')
    item=base_normalize(''.join(part))
    if item: out.append(item)
    return out

def parse_ingredients(value):
    if isinstance(value, list): sections=value
    elif isinstance(value, str):
        text=value.strip()
        if not text: return [], 'missing'
        if text.startswith('['):
            try: sections=ast.literal_eval(text)
            except (ValueError, SyntaxError): return [], 'invalid_list'
        else: sections=[text]
    else: return [], 'missing'
    if not isinstance(sections,list) or any(not isinstance(x,str) for x in sections): return [], 'invalid_section'
    sections=[s.strip() for s in sections if s.strip()]
    if not sections: return [], 'missing'
    # Keep the ordered main formula and omit optional shade colorants.
    # Do not assign ranks to 'may contain' pigments. Export for review.
    had_optional=any(re.search(r'\bmay\s+contain\b', s, re.I) for s in sections)
    if had_optional:
        kept=[]
        for section in sections:
            before=re.split(r'\bmay\s+contain\b',section,maxsplit=1,flags=re.I)[0].strip(' .;')
            if before: kept.append(before)
            if re.search(r'\bmay\s+contain\b',section,re.I): break
        sections=kept
        if not sections: return [], 'may_contain_review'
    if len(sections)==1: text,status=sections[0], 'single'
    else:
        # Hold separate phases and other labeled sections for review.
        labels=[bool(re.match(r'\s*[^,]{1,80}:',s)) for s in sections]
        formulas=[i for i,s in enumerate(sections) if s.count(',')>=4]
        if len(formulas)==1 and all(labels[i] and sections[i].count(',')<4 for i in range(len(sections)) if i!=formulas[0]):
            text,status=sections[formulas[0]],'selected_formula_review'
        elif any(labels): return [], 'labeled_sections_review'
        else: text,status=' '.join(sections),'joined_fragments_review'
    if had_optional: status = status + '_optional_colorants_review'
    try: return split_formula(text),status
    except ValueError: return [],'delimiter_review'

products['normalized_highlights'] = products.highlights.apply(highlights)
parsed=products.ingredients.apply(parse_ingredients)
products['ingredients_list'] = parsed.map(lambda x:x[0])
products['parse_status'] = parsed.map(lambda x:x[1])
products['normalized_ingredients'] = products.ingredients_list.map(lambda xs:list(dict.fromkeys(normalize(x) for x in xs if normalize(x))))
print(products.parse_status.value_counts().to_string())
assert split_formula('Water, 1,2-Hexanediol, Glycerin') == ['water','1,2-hexanediol','glycerin']
assert parse_ingredients(['Water, Glycerin, Pentylene','Glycol, Panthenol'])[0][-2] == 'pentylene glycol'
assert highlights(float('nan')) == []
assert parse_ingredients(['Water, Glycerin. May Contain: Iron Oxides'])[0] == ['water', 'glycerin']

<unknown>:1: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.


parse_status
single                                               5986
labeled_sections_review                               632
single_optional_colorants_review                      402
delimiter_review                                      201
joined_fragments_review                               186
selected_formula_review_optional_colorants_review      65
selected_formula_review                                46
joined_fragments_review_optional_colorants_review      31


In [3]:
products.loc[products.parse_status.str.contains('review'), ['product_id','product_name','ingredients','ingredients_list','parse_status']].to_csv('ingredient_parse_review_new.csv', index=False)
def blob(s): return len(s)>160 or len(s.split())>20 or bool(re.search(r'\b(?:ingredients|directions|how to use)\s*:', s))
blobs=[(r.product_id,r.product_name,x) for r in products.itertuples() for x in r.normalized_ingredients if blob(x)]
pd.DataFrame(blobs,columns=['product_id','product_name','ingredient']).to_csv('ingredient_blob_review_new.csv',index=False)
skincare_scope=products.primary_category.eq('Skincare') & products.secondary_category.isin(CATEGORIES)
makeup_scope=products.primary_category.eq('Makeup') & products.tertiary_category.isin(MAKEUP_TYPES)
model=products[(skincare_scope | makeup_scope) & products.normalized_ingredients.map(bool)].copy()
model['price_usd']=pd.to_numeric(model.price_usd,errors='coerce')
model=model.drop_duplicates('product_id').reset_index(drop=True)
print('Eligible products:',len(model))
print('By category:\n',model.primary_category.value_counts().to_string())
print('Eligible missing raw highlights:',model.highlights.isna().sum())
print('Status breakdown:\n',model.parse_status.value_counts().to_string())
def singleton(lists):
    c=Counter(x for row in lists for x in set(row))
    return {'vocabulary':len(c),'singletons':sum(v==1 for v in c.values()),'rate':sum(v==1 for v in c.values())/len(c) if c else 0}
print('Before aliases/percentage cleanup:',singleton(model.ingredients_list))
print('After aliases/percentage cleanup:',singleton(model.normalized_ingredients))

Eligible products: 3307
By category:
 primary_category
Skincare    1879
Makeup      1428
Eligible missing raw highlights: 648
Status breakdown:
 parse_status
single                                               2820
single_optional_colorants_review                      359
joined_fragments_review                                75
selected_formula_review_optional_colorants_review      22
joined_fragments_review_optional_colorants_review      16
selected_formula_review                                15
Before aliases/percentage cleanup: {'vocabulary': 7418, 'singletons': 3586, 'rate': 0.4834187112429226}
After aliases/percentage cleanup: {'vocabulary': 7253, 'singletons': 3476, 'rate': 0.47924996553150423}


## Review texture
The threshold is ten nonempty review texts. Mentions use a small vocabulary and simple clause-level negation; manually validate exported examples. A missing mention is not an assertion that a texture is absent.

In [4]:
TERMS={'lightweight':r'lightweight|light weight','heavy':r'heavy|thick','greasy':r'greasy|oily','sticky':r'sticky|tacky','watery':r'watery|runny','creamy':r'creamy'}
PATTERNS={k:re.compile(r'\b(?:'+v+r')\b',re.I) for k,v in TERMS.items()}
CLAUSE=re.compile(r'[.!?;]+|\b(?:but|however|although)\b',re.I)
NEG=re.compile(r"\b(?:not|never|isn't|wasn't|doesn't|didn't|no)\b",re.I)
texture_cols=[f'{k}_{p}' for k in TERMS for p in ('pos','neg')]
def extract_texture(text):
    out=dict.fromkeys(texture_cols,0)
    for clause in CLAUSE.split(unicodedata.normalize('NFKC',text).casefold()):
        for key,pattern in PATTERNS.items():
            for match in pattern.finditer(clause):
                # Exclude common descriptions of skin type, not product texture.
                if key == 'greasy' and re.match(r'\s+(?:skin|skin type)\b', clause[match.end():]):
                    continue
                prefix=' '.join(clause[:match.start()].split()[-4:])
                out[f"{key}_{'neg' if NEG.search(prefix) else 'pos'}"]=1
    return out
assert extract_texture('not sticky, but creamy')['sticky_neg'] == 1
assert extract_texture('not sticky, but creamy')['creamy_pos'] == 1
assert extract_texture('helps control oily skin')['greasy_pos'] == 0

files=sorted(DATA_DIR.glob('reviews_*.csv'))
if not files: raise FileNotFoundError('No review files found')
ids=set(model.product_id)
frames=[]
for path in files:
    for chunk in pd.read_csv(path,usecols=['product_id','author_id','submission_time','review_text'],chunksize=100000,low_memory=False):
        chunk=chunk[chunk.product_id.isin(ids)].copy()
        chunk['review_text']=chunk.review_text.fillna('').astype(str).str.strip()
        chunk=chunk[chunk.review_text.ne('')]
        if not chunk.empty: frames.append(chunk)
reviews=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame(columns=['product_id','author_id','submission_time','review_text'])
reviews=reviews.drop_duplicates(['product_id','author_id','submission_time','review_text']).reset_index(drop=True)
texture=pd.DataFrame(reviews.review_text.map(extract_texture).tolist(),index=reviews.index,columns=texture_cols).fillna(0)
reviews=pd.concat([reviews,texture],axis=1)
grouped=reviews.groupby('product_id')
counts=grouped.size().rename('usable_review_count')
sums=grouped[texture_cols].sum()
product_texture=sums.div(counts,axis=0).join(counts).reset_index()
model=model.merge(product_texture,on='product_id',how='left')
model['usable_review_count']=model.usable_review_count.fillna(0).astype(int)
model[texture_cols]=model[texture_cols].fillna(0)
model['texture_available']=model.usable_review_count.ge(10) & model[texture_cols].sum(axis=1).gt(0)
print('Usable review rows:',len(reviews))
print('Products with >=10 reviews:',model.usable_review_count.ge(10).sum())
print('Products with >=10 reviews and texture mentions:',model.texture_available.sum())
print('Products with <10 reviews:',model.usable_review_count.lt(10).sum())
print('Review coverage by category:\n',model.groupby('primary_category').agg(products=('product_id','size'), at_least_10=('usable_review_count',lambda x:x.ge(10).sum()), texture_available=('texture_available','sum')).to_string())
mentioned=reviews[texture_cols].sum(axis=1).gt(0)
reviews.loc[mentioned,['product_id','review_text']+texture_cols].sample(n=min(200,int(mentioned.sum())),random_state=42).to_csv('texture_review_validation_new.csv',index=False)

Usable review rows: 947814
Products with >=10 reviews: 1723
Products with >=10 reviews and texture mentions: 1702
Products with <10 reviews: 1584
Review coverage by category:
                   products  at_least_10  texture_available
primary_category                                          
Makeup                1428            0                  0
Skincare              1879         1723               1702


## Price and feature matrices
Package price determines eligibility. Unit price is shown only for unambiguous comparable quantities; oz without an explicit volume/weight indicator is left unresolved.

In [5]:
def parse_size(value):
    if not isinstance(value,str) or not value.strip(): return np.nan,None,'missing'
    s=unicodedata.normalize('NFKC',value).casefold().strip()
    if re.search(r'\b(?:set|kit|pack|pieces|pcs)\b|\d+\s*[x×]\s*\d',s): return np.nan,None,'multipack_or_kit'
    if re.search(r'\d\s*(?:-|–|to)\s*\d',s): return np.nan,None,'range'
    for pattern,unit,mult in [(r'(?<!\d)(\d*\.?\d+)\s*ml\b','ml',1),(r'(?<!\d)(\d*\.?\d+)\s*(?:g|grams?)\b','g',1),(r'(?<!\d)(\d*\.?\d+)\s*fl\.?\s*oz\.?','ml',29.5735)]:
        matches=re.findall(pattern,s)
        if len(matches)>1: return np.nan,None,'multiple_sizes'
        if matches:
            amount=float(matches[0])*mult
            return (amount,unit,'parsed') if amount>0 else (np.nan,None,'invalid')
    return np.nan,None,'ambiguous'
size=pd.DataFrame(model['size'].map(parse_size).tolist(),columns=['size_amount','size_dimension','size_status'])
model=pd.concat([model,size],axis=1)
model['unit_price']=np.where(model.price_usd.gt(0)&model.size_amount.gt(0),model.price_usd/model.size_amount,np.nan)
assert parse_size('2 oz/ 60 mL')[:2] == (60.0,'ml')
print(model.size_status.value_counts().to_string())

sets=[set(x) for x in model.normalized_ingredients]
vectorizer=TfidfVectorizer(analyzer=lambda x:x,lowercase=False)
X_tfidf=vectorizer.fit_transform(model.normalized_ingredients)
def rank_dict(xs): return {x:1/np.log2(i+2) for i,x in reversed(list(enumerate(xs)))}
rank_vectorizer=DictVectorizer(sparse=True)
X_rank=TfidfTransformer().fit_transform(rank_vectorizer.fit_transform(model.normalized_ingredients.map(rank_dict)))
encoder=MultiLabelBinarizer(sparse_output=True)
X_high=encoder.fit_transform(model.normalized_highlights)
positions={pid:i for i,pid in enumerate(model.product_id)}
print('Feature matrices:',X_tfidf.shape,X_rank.shape,X_high.shape)

size_status
parsed              2949
missing              174
ambiguous            170
multipack_or_kit      11
multiple_sizes         3
Feature matrices: (3307, 7253) (3307, 7253) (3307, 77)


## Recommendation systems
All systems use the same ingredient overlap, category, and lower-package-price filters. Rank-weighted systems put more emphasis on earlier ingredients as a proxy for concentration; actual concentrations are unknown. The full system uses review texture when both products have usable texture evidence, otherwise highlights when both have them, otherwise ingredients alone. Its weights are initial settings, not validated values.

In [6]:
def recommend(product_id, system='full', top_k=5, ingredient_weight=0.7, different_brand=False):
    if system not in {'jaccard','tfidf','rank','full'}: raise ValueError('Unknown system')
    if product_id not in positions: raise ValueError('Product not in model')
    if not isinstance(top_k,int) or top_k<1: raise ValueError('top_k must be positive')
    if not 0<ingredient_weight<1: raise ValueError('ingredient_weight must be between 0 and 1')
    i=positions[product_id]; q=model.iloc[i]
    if pd.isna(q.price_usd) or q.price_usd<=0: raise ValueError('Query needs valid price')
    if system=='jaccard':
        scores=np.array([len(sets[i]&s)/len(sets[i]|s) if sets[i]|s else 0 for s in sets])
    else:
        matrix=X_tfidf if system=='tfidf' else X_rank
        scores=cosine_similarity(matrix[i],matrix).ravel()
    used=np.full(len(model),system,dtype=object)
    if system=='full':
        hl=cosine_similarity(X_high[i],X_high).ravel() if X_high.shape[1] else np.zeros(len(model))
        use_h=(X_high.getnnz(axis=1)>0) & (X_high[i].nnz>0)
        # Texture compares signed positive/negated mention rates only for
        # attributes mentioned by both products. No shared evidence -> fallback.
        vals=model[texture_cols].to_numpy(dtype=float).reshape(len(model),len(TERMS),2)
        qv=vals[i]; both=(vals.sum(axis=2)>0)&(qv.sum(axis=1)>0)
        distance=np.abs((vals[:,:,0]-vals[:,:,1])-(qv[:,0]-qv[:,1]))
        texture=np.divide((1-np.minimum(distance,1))*both,1).sum(axis=1)
        n=both.sum(axis=1)
        texture=np.divide(texture,n,out=np.zeros(len(model)),where=n>0)
        use_t=model.texture_available.to_numpy() & bool(q.texture_available) & (n>0)
        use_h &= ~use_t
        scores[use_h]=ingredient_weight*scores[use_h]+(1-ingredient_weight)*hl[use_h]
        scores[use_t]=ingredient_weight*scores[use_t]+(1-ingredient_weight)*texture[use_t]
        used[:]='ingredients_only'; used[use_h]='ingredients_and_highlights'; used[use_t]='ingredients_and_texture'
    mask=model.primary_category.eq(q.primary_category) & model.secondary_category.eq(q.secondary_category)
    mask &= model.tertiary_category.eq(q.tertiary_category) if pd.notna(q.tertiary_category) else model.tertiary_category.isna()
    # Setting sprays and setting powders share a dataset category but are different forms.
    if q.primary_category == 'Makeup' and q.tertiary_category == 'Setting Spray & Powder':
        def setting_form(name):
            s=str(name).casefold()
            if 'spray' in s or 'mist' in s: return 'spray'
            if 'powder' in s: return 'powder'
            return 'unknown'
        form=setting_form(q.product_name)
        mask &= model.product_name.map(setting_form).eq(form) if form != 'unknown' else False
    mask &= model.price_usd.gt(0)&model.price_usd.lt(q.price_usd)&model.product_id.ne(product_id)
    if different_brand: mask &= model.brand_name.ne(q.brand_name)
    mask &= np.array([bool(sets[i]&s) for s in sets])
    idx=np.flatnonzero(mask.to_numpy()); result=model.iloc[idx].copy()
    result['similarity']=scores[idx]; result['features_used']=used[idx]
    result['shared_ingredients']=[sorted(sets[i]&sets[j]) for j in idx]
    result['package_savings_usd']=q.price_usd-result.price_usd
    result['unit_price_comparable']=pd.notna(q.unit_price)&result.unit_price.notna()&result.size_dimension.eq(q.size_dimension)
    result['unit_price_savings']=np.where(result.unit_price_comparable,q.unit_price-result.unit_price,np.nan)
    cols=['product_id','product_name','primary_category','brand_name','size','price_usd','unit_price','size_dimension','unit_price_comparable','unit_price_savings','similarity','features_used','usable_review_count','package_savings_usd','shared_ingredients']
    return result.sort_values(['similarity','price_usd','product_id'],ascending=[False,True,True]).head(top_k)[cols].reset_index(drop=True)

query_id=model.loc[model.price_usd.gt(0),'product_id'].iloc[0]
print('Example query:',query_id,model.loc[model.product_id.eq(query_id),'product_name'].iloc[0])
for system in ['jaccard','tfidf','rank','full']:
    print('\n',system)
    display(recommend(query_id,system=system,top_k=3))

Example query: P398965 Rose Lip Conditioner

 jaccard


,product_id,product_name,primary_category,brand_name,size,price_usd,unit_price,size_dimension,unit_price_comparable,unit_price_savings,similarity,features_used,usable_review_count,package_savings_usd,shared_ingredients



 tfidf


,product_id,product_name,primary_category,brand_name,size,price_usd,unit_price,size_dimension,unit_price_comparable,unit_price_savings,similarity,features_used,usable_review_count,package_savings_usd,shared_ingredients



 rank


,product_id,product_name,primary_category,brand_name,size,price_usd,unit_price,size_dimension,unit_price_comparable,unit_price_savings,similarity,features_used,usable_review_count,package_savings_usd,shared_ingredients



 full


,product_id,product_name,primary_category,brand_name,size,price_usd,unit_price,size_dimension,unit_price_comparable,unit_price_savings,similarity,features_used,usable_review_count,package_savings_usd,shared_ingredients


## Evaluation
Recommendation quality needs independent labels; do not fabricate a dupe list or interpret coverage as accuracy.

In [10]:
def evaluate_labels(labels,system,k=5):
    required={'query_product_id','candidate_product_id','relevance'}
    if not required.issubset(labels): raise ValueError('Missing label columns')
    if labels.duplicated(['query_product_id','candidate_product_id']).any(): raise ValueError('Duplicate labeled pairs')
    if not labels.relevance.isin([0,1]).all(): raise ValueError('Relevance must be 0 or 1')
    rows=[]
    for query,group in labels.groupby('query_product_id'):
        if query not in positions: continue
        try: rec=recommend(query,system=system,top_k=k)
        except ValueError: continue
        mapping=dict(zip(group.candidate_product_id,group.relevance)); returned=rec.product_id.tolist()
        judged=[x for x in returned if x in mapping]; hits=sum(mapping[x] for x in judged); relevant=sum(mapping.values())
        rows.append({'query_product_id':query,'returned':len(returned),'judged':len(judged),'unjudged':len(returned)-len(judged),'precision_judged':hits/len(judged) if judged else np.nan,'recall_known':hits/relevant if relevant else np.nan,'coverage':int(bool(returned))})
    return pd.DataFrame(rows)

label_path=Path('dupe_labels.csv')
if label_path.exists():
    labels=pd.read_csv(label_path)
    for system in ['jaccard','tfidf','rank','full']:
        result=evaluate_labels(labels,system)
        print(system, result[['precision_judged','recall_known','coverage','unjudged']].mean().to_dict() if len(result) else 'no evaluable queries')
else:
    print('No dupe_labels.csv found: relevance and model quality cannot yet be measured.')
    print('Required columns: query_product_id,candidate_product_id,relevance (0 or 1).')

No dupe_labels.csv found: relevance and model quality cannot yet be measured.
Required columns: query_product_id,candidate_product_id,relevance (0 or 1).


In [11]:
# Smoke checks for model execution and recommendation constraints.
# These do not establish whether recommendations are good dupes.
testable=model.loc[model.price_usd.gt(0)].head(25)
for q in testable.itertuples():
    for system in ['jaccard','tfidf','rank','full']:
        found=recommend(q.product_id,system=system,top_k=5)
        assert (found.price_usd < q.price_usd).all()
        assert q.product_id not in set(found.product_id)
        assert found.similarity.between(0,1.0000001).all()
        if len(found):
            candidates=model.set_index('product_id').loc[found.product_id]
            assert candidates.secondary_category.eq(q.secondary_category).all()
missing=model.loc[model.price_usd.gt(0) & model.normalized_highlights.map(lambda x: not x)]
if not missing.empty:
    q=missing.iloc[0]
    found=recommend(q.product_id,system='full')
    assert not found.features_used.eq('ingredients_and_highlights').any()
    print('Missing-highlight query ran:',q.product_id,'recommendations:',len(found))
print('Smoke checks passed for',len(testable),'queries and all four systems.')
for category in ['Skincare','Makeup']:
    queries=model[model.primary_category.eq(category) & model.price_usd.gt(0)].head(20)
    for q in queries.itertuples():
        for system in ['jaccard','tfidf','rank','full']:
            result=recommend(q.product_id,system=system)
            assert result.primary_category.eq(category).all()
    print(category, 'queries tested:',len(queries))
assert rank_dict(['first','second','third'])['first'] > rank_dict(['first','second','third'])['second'] > rank_dict(['first','second','third'])['third']


Missing-highlight query ran: P467602 recommendations: 5
Smoke checks passed for 25 queries and all four systems.
Skincare queries tested: 20
Makeup queries tested: 20


In [12]:
# Inspect the ingredient-order weights for any product.
# These are model weights, not measured concentrations.
def show_ingredient_order(product_id):
    if product_id not in positions:
        raise ValueError('Product not in model')
    ingredients=model.iloc[positions[product_id]].normalized_ingredients
    weights=rank_dict(ingredients)
    return pd.DataFrame({'position':range(1,len(ingredients)+1), 'ingredient':ingredients, 'rank_weight':[weights[x] for x in ingredients]})

makeup_examples=model[model.primary_category.eq('Makeup') & model.tertiary_category.eq('Foundation') & model.price_usd.gt(0)]
if not makeup_examples.empty:
    makeup_id=makeup_examples.iloc[0].product_id
    print('Makeup example:',makeup_examples.iloc[0].product_name, makeup_id)
    display(show_ingredient_order(makeup_id).head(10))
    display(recommend(makeup_id,system='full',top_k=5))


Makeup example: Stick Foundation P410504


,position,ingredient,rank_weight
0,1,phenyl trimethicone,1.000000
1,2,dimethicone,0.630930
2,3,isononyl isononanoate,0.500000
3,4,silica,0.430677
4,5,bis-diglyceryl polyacyladipate-2,0.386853
5,6,ozokerite,0.356207
6,7,polymethyl methacrylate,0.333333
7,8,triisodecyl trimellitate,0.315465
8,9,polyethylene,0.301030
9,10,microcrystalline wax/cera microcristallina,0.289065


,product_id,product_name,primary_category,brand_name,size,price_usd,unit_price,size_dimension,unit_price_comparable,unit_price_savings,similarity,features_used,usable_review_count,package_savings_usd,shared_ingredients
0,P454990,Mini babassu foundcealer skincare foundation S...,Makeup,tarte,0.33 oz/ 10 mL,16.0,1.600000,ml,False,NaN,0.351075,ingredients_only,0,9.0,"[bis-diglyceryl polyacyladipate-2, dimethicone..."
1,P469460,Vision Cream Cover Adjustable Foundation & Con...,Makeup,Danessa Myricks Beauty,0.34 oz/ 10 mL,22.0,2.200000,ml,False,NaN,0.228082,ingredients_only,0,3.0,"[ethylhexyl palmitate, ozokerite, phenyl trime..."
2,P481695,Yummy Skin Serum Foundation Refill,Makeup,Danessa Myricks Beauty,0.85 oz/ 25mL,24.0,0.960000,ml,False,NaN,0.208116,ingredients_only,0,1.0,"[ethylhexyl palmitate, limnanthes alba (meadow..."
3,P447557,Lock-It Powder Foundation Refills,Makeup,KVD Beauty,0.31 oz/ 9 g,24.0,2.666667,g,True,0.080586,0.138426,ingredients_only,0,1.0,"[dimethicone, kaolin, phenyl trimethicone]"
4,P447121,Hello Happy Flawless Brightening Foundation SP...,Makeup,Benefit Cosmetics,0.33 oz/ 10.0 mL,15.0,1.500000,ml,False,NaN,0.127311,ingredients_only,0,10.0,"[dimethicone, phenyl trimethicone, silica]"
